# Word Embeddings

Dans cette démonstration, nous allons voir comment utiliser des word embeddings pré-appris avec [`spacy`](https://spacy.io/) et [`gensim`](https://radimrehurek.com/gensim/).

Nous verrons ensuite comment apprendre nos propres embeddings.

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 9.5 MB/s eta 0:00:00


In [2]:
import itertools
import os
import pathlib
import sys
import re
import typing

import gensim.models
import gensim.models.phrases
import spacy
import warnings

warnings.filterwarnings("ignore")

## Modèle pré-entraîné de Spacy

Nous allons travailler avec `en_core_web_lg`, un grand modèle pour l'anglais (382 Mo), qui relie 684 830 mots à 342 918 vecteurs de 300 dimensions.

In [3]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.9 MB/s  0:01:19
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


## Utilisation

In [4]:
nlp = spacy.load("en_core_web_lg")

doc = nlp("This is some text that I am processing with Spacy")

print(doc[3].vector)

print("Vecteur moyen")
print(doc.vector)

[ 0.037103  -0.31259   -0.17857    0.30001    0.078154   0.17958
  0.12048   -0.11879   -0.20601    1.2849    -0.20409    0.80613
  0.34344   -0.19191   -0.084511   0.17339    0.042483   2.0282
 -0.16278   -0.60306   -0.53766    0.35711    0.22882    0.1171
  0.42983    0.16165    0.407      0.036476   0.52636   -0.13524
 -0.016897   0.029259  -0.079115  -0.32305    0.052255  -0.3617
 -0.18355   -0.34717   -0.3691     0.16881    0.21018   -0.38376
 -0.096909  -0.36296   -0.37319    0.0021152  0.32512    0.063977
  0.36249   -0.26935   -0.59341   -0.13625    0.016425  -0.2474
 -0.07498    0.034708  -0.01476   -0.11648    0.25559   -0.35002
 -0.52707    0.21221    0.062456   0.26184    0.53149    0.34957
 -0.22692    0.44076    0.4438     0.6335    -0.049757  -0.08134
  0.65618   -0.4716     0.090675  -0.084873   0.31455   -0.38495
 -0.19247    0.48064    0.26688    0.095743   0.13024    0.37023
  0.46269   -0.32844    0.17375   -0.36325    0.30672   -0.075042
 -0.64684   -0.49822    0.1

## Modèle pré-entraîné de Gensim

Gensim ne fournit pas d'embeddings pré-entraînés, mais un mécanisme pour en charger.

Quelques sources d'embeddings pré-entraînés :

- [Documentation word2vec de Google News](https://code.google.com/archive/p/word2vec/) ; [GoogleNews-vectors-negative300.bin.gz](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)

- [Dépôt de word embeddings du NLPL](http://vectors.nlpl.eu/repository/)

- [fasttext](https://fasttext.cc/docs/en/crawl-vectors.html)

- [Embeddings français de Jean-Philippe Fauconnier](https://fauconnier.github.io/)

Nous utilisons le modèle français de ce dernier dépôt (CBOW, dimension 200, cutoff 100, ni lemmatisation, ni étiquetage morphosyntaxique, ni groupes de mots).

Ce modèle est l'œuvre de [Jean-Philippe Fauconnier](https://fauconnier.github.io/), sous licence [CC BY 3.0](https://creativecommons.org/licenses/by/3.0/deed.fr) : nous en téléchargeons une copie inchangée depuis `shuuchuu/datasets`, où il est coupé en deux morceaux, car GitHub refuse les fichiers de plus de 100 Mio. `cat` les rassemble.

In [5]:
# Le modèle est coupé en deux morceaux dans shuuchuu/datasets
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh frwac-word2vec
!cat frwac-word2vec/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin.part-* > frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin

✓ frwac-word2vec — 121M


In [6]:
!ls -l 
model_file = "frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin"

total 123104
-rw-r--r-- 1 root root 126052447 Nov  5 18:37 frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin
drwxr-xr-x 2 root root      4096 Nov  5 18:37 frwac-word2vec


## Utilisation de [`gensim.models.KeyedVectors`](https://radimrehurek.com/gensim/models/keyedvectors.html)

In [7]:
model = gensim.models.KeyedVectors.load_word2vec_format(
  model_file, binary=model_file.endswith(".bin")
)
print("Taille du vocabulaire : ", len(model.key_to_index))

Taille du vocabulaire :  155562


In [8]:
for i, word in enumerate(model.index_to_key):
  print(word)
  if i > 50:
    break

</s>
de
la
et
le
l'
les
à
des
d'
en
du
un
est
une
pour
dans
par
sur
qui
que
il
au
a
pas
ce
ou
avec
plus
je
vous
ne
se
sont
s'
qu'
n'
aux
on
mais
nous
c'
son
cette
comme
être
elle
tout
j'
ont
si
ces


In [9]:
# Récupère le vecteur d'un mot
vector = model["facile"]

# Le prétraitement doit être le même que pendant l'entraînement
words = "Ceci est une phrase transformée à l'aide de Gensim".lower().split(" ")
vectors = [(word, model[word]) for word in words if word in model.key_to_index]
print("Mots transformés :")
for word, vector in vectors:
  print(word)

Mots transformés :
ceci
est
une
phrase
transformée
à
l'aide
de


In [10]:
model.most_similar("berlin")

[('dresde', 0.7241218090057373),
 ('munich', 0.681473433971405),
 ('leipzig', 0.6640617251396179),
 ('hambourg', 0.6593071818351746),
 ('bonn', 0.6533075571060181),
 ('allemagne', 0.6425371766090393),
 ('düsseldorf', 0.6412529945373535),
 ('allemande', 0.6355056166648865),
 ('potsdam', 0.6336090564727783),
 ('strasse', 0.6298079490661621)]

In [11]:
print(model.similarity("berlin", "munich"))
print(model.similarity("berlin", "paris"))

0.68147343
0.18069758


## Opérations dans l'espace sémantique

À l'aide de [`gensim.models.keyedvectors.KeyedVectors.similar_by_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector), nous allons définir une fonction d'analogie :

`x` est à `y` ce que `z` est à ?

*Ex.* : *Roi* est à *Reine* ce que *Oncle* est à ?

Réponse : *Tante*

In [12]:
def semantic_analogy(x, y, z):
  return model.similar_by_vector(model[y] - model[x] + model[z])

In [13]:
semantic_analogy("homme", "femme", "chien")

[('chien', 0.8191524744033813),
 ('chienne', 0.7883859872817993),
 ('chatte', 0.6961268186569214),
 ('toutou', 0.6323156952857971),
 ('chiot', 0.6281698346138),
 ('adorable', 0.6145821809768677),
 ('sociable', 0.599670946598053),
 ('chiens', 0.5982583165168762),
 ('affectueux', 0.5909204483032227),
 ('chats', 0.5903921127319336)]

## Apprentissage de word embeddings

### Le corpus 20 Newsgroups

20 000 « posts » répartis en 20 thématiques.

In [14]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

--2026-11-05 18:37:05--  https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz [following]
--2026-11-05 18:37:06--  https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14464277 (14M) [application/octet-stream]
Saving to: ‘20news-bydate.tar.gz’

20news-bydate.tar.g 100%[===================>]  13.79M  4.71MB/s    in 2.9s    

2026-11-05 18:37:11 (4.71

In [15]:
!ls 20news-bydate-test/

alt.atheism		  rec.autos	      sci.space
comp.graphics		  rec.motorcycles     soc.religion.christian
comp.os.ms-windows.misc   rec.sport.baseball  talk.politics.guns
comp.sys.ibm.pc.hardware  rec.sport.hockey    talk.politics.mideast
comp.sys.mac.hardware	  sci.crypt	      talk.politics.misc
comp.windows.x		  sci.electronics     talk.religion.misc
misc.forsale		  sci.med


In [16]:
contents = [
  p.read_text(encoding="latin-1") for p in pathlib.Path(".").glob("20news-bydate-*/*/*")
]


print(f"{len(contents)} textes récupérés")

18846 textes récupérés


In [17]:
print(contents[1])

From: jagst18+@pitt.edu (Josh A Grossman)
Subject: Re: Dillon puts foot in mouth, film at 11
Organization: The Zets at the University of Pittsburgh (aka The Pitt)
Lines: 17


I have also heard about HCI claiming thant anyone they  get an address
from is a member.  If this is the case their membership rolls are
grossly inflated and we should not call them and give them a name
and address to add to their already false rolls.  Perhaps
if you could get a copy of their existing membership, then pretend to
be an existing member, do that several thousand times, you could
hurt HCI.  But names are power.  Remeber the NRA uses the fact
that it has 3 million paid members in order to flex its muscles.

Perhaps politicians don't realize the lying tactics of HCI, wait a
minute, HCI learned it from politicians....

Later,
Josh





In [18]:
# Nettoyage
def preprocess(content: str) -> typing.List[str]:
  offset = content.find("\n\n")
  if offset > 0:
    content = content[offset + 2 :]
  sentences = (
    re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
    for line in content.splitlines()
    if not line.endswith("writes:")
  )
  return list(itertools.chain.from_iterable(sentences))


texts = [preprocess(content) for content in contents]

In [19]:
print(texts[0])

['Do', 'you', 'know', 'how', 'many', 'deaths', 'each', 'year', 'are', 'caused', 'by', 'self', 'inflicted', 'gun', 'shot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters', 'There', 'are', 'roughly', '1200', 'fatal', 'firearms', 'related', 'accidents', 'each', 'year', 'The', 'large', 'majority', 'involve', 'rifles', 'and', 'shotgun', 'there', 'are', 'under', '500', 'fatal', 'handgun', 'accidents', 'each', 'year', 'I', 'really', 'doubt', 'all', 'of', 'those', 'occur', 'while', 'the', 'pistol', 'is', 'holstered', 'so', 'the', 'number', 'of', 'self', 'inflicted', 'gunshot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters', 'is', 'probably', 'well', 'under', '250', 'per', 'year', "I'm", 'neither', 'a', 'doctor', 'nor', 'a', 'firearms', 'tech', 'expert', 'but', 'it', 'would', 'seem', 'that', 'given', 'the', 'way', 'a', 'holstered', 'gun', 'points', 'accidental', 'injuries', 'inflicted', 'that', 'way', 'would', 'be', 'among', 'the', 'least', 'lethal', 'David', 'Veal', 'Univ', 

## Détection de groupes de mots avec le Phraser de Gensim

In [20]:
common_terms = ["of", "with", "without", "and", "or", "the", "a"]

phrases = gensim.models.phrases.Phrases(texts, connector_words=frozenset(common_terms))

phraser = gensim.models.phrases.Phraser(phrases)
phrased_texts = list(phraser[texts])
len(phrased_texts)

18846

In [21]:
model = gensim.models.Word2Vec(
  phrased_texts,
  min_count=3,
  vector_size=200,
  workers=2,
  window=5,
  epochs=30,
).wv

In [22]:
model.most_similar("New_York")

[('City', 0.4925885796546936),
 ('Times', 0.44712013006210327),
 ('California', 0.43223416805267334),
 ('Los_Angeles', 0.42236074805259705),
 ('7th', 0.4212449789047241),
 ('Atlanta', 0.4197428226470947),
 ('Toronto', 0.4174637794494629),
 ('Philadelphia', 0.41541552543640137),
 ('Culture', 0.41309940814971924),
 ('November', 0.41074812412261963)]

In [23]:
semantic_analogy("wind", "fly", "water")

[('water', 0.6909886002540588),
 ('fly', 0.6556036472320557),
 ('drain', 0.40905746817588806),
 ('fuel', 0.40162205696105957),
 ('drop', 0.4000636637210846),
 ('heat', 0.3958766460418701),
 ('generate', 0.3953576385974884),
 ('move', 0.3918507695198059),
 ('air', 0.3912471830844879),
 ('discharge', 0.39114677906036377)]